# Lesson 4 — Embedding & Clustering: The Canonical Mechanism Map

**Learning objective:** Understand how PTCs are embedded into a shared semantic space, clustered into canonical mechanisms, cross-mapped between management and press corpora, and sector-tagged. Walk through the full 4-axis typology with synthetic vectors — no AWS calls needed.

---

## Why embed and cluster?

After extraction, the pool contains hundreds of PTCs like:
- *"Red Sea disruptions elevated logistics costs by €45M"*
- *"Shipping route disruptions added freight costs"*
- *"Middle East tensions increased our transport expenses"*

These are three different wordings of the same underlying mechanism. Clustering collapses them into one **canonical mechanism** — a representative description that is used in all downstream analysis. Without this step, you'd run Granger on hundreds of near-duplicate sparse series.

---

## The embedding text format

Before embedding, each PTC is formatted as:
```
[direction] [polarity] mechanism text
```
e.g.: `"[precursor] [-1] Red Sea disruptions elevated logistics costs by €45M..."`

This ensures that opposite-direction or opposite-polarity variants of the same mechanism are embedded slightly apart in the vector space, making the direction × polarity split in Phase B meaningful.

In [ ]:
import sys
import json
import tempfile
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, '../../src')

from signal_intelligence.clustering import (
    _hdbscan_cluster, _split_by_direction_polarity,
    _cosine_sim, _cross_map, cluster_ptcs
)

print("Imports OK")

## Exercise 1 — Cosine similarity intuition

Cosine similarity measures the angle between vectors. 1.0 = identical direction, 0.0 = orthogonal, −1.0 = opposite.

In [ ]:
rng = np.random.default_rng(42)

def unit(v): return v / np.linalg.norm(v)

base = unit(rng.standard_normal(16).astype(np.float32))

# Near-identical (same mechanism, slightly different phrasing)
near = unit(base + rng.standard_normal(16).astype(np.float32) * 0.05)

# Unrelated mechanism
unrelated = unit(rng.standard_normal(16).astype(np.float32))

print(f"Same mechanism  (noisy copy):  cosine = {_cosine_sim(base, near):.4f}")
print(f"Unrelated mechanism:           cosine = {_cosine_sim(base, unrelated):.4f}")

## Exercise 2 — HDBSCAN clusters tight groups

In [ ]:
# Build 3 synthetic clusters (logistics, forex, rubber) with 4 PTCs each
rng = np.random.default_rng(0)

cluster_centers = [
    unit(rng.standard_normal(32).astype(np.float32)),  # logistics
    unit(rng.standard_normal(32).astype(np.float32)),  # forex
    unit(rng.standard_normal(32).astype(np.float32)),  # rubber
]
mechanism_names = ["logistics", "forex", "rubber"]

vectors = []
true_labels = []
for label, center in enumerate(cluster_centers):
    for _ in range(4):
        v = center + rng.standard_normal(32).astype(np.float32) * 0.02
        vectors.append(unit(v))
        true_labels.append(label)

V = np.stack(vectors)
predicted = _hdbscan_cluster(V, min_cluster_size=3)

print("True labels:     ", true_labels)
print("HDBSCAN labels:  ", predicted.tolist())
print(f"Unique clusters: {len(set(predicted[predicted >= 0]))} (expected 3)")
print(f"Noise points:    {(predicted == -1).sum()} (should be 0 for tight clusters)")

## Exercise 3 — Direction × polarity split

After HDBSCAN, a cluster might contain PTCs with mixed direction or polarity. We split them: a "rubber costs" cluster should not merge a `consequence -1` (rubber already hurt Q3) with a `precursor -1` (rubber will hurt Q4).

In [ ]:
# Simulate a mixed cluster: same mechanism, two temporal stances
mixed_df = pd.DataFrame([
    {"_label": 0, "direction": "precursor",   "polarity": -1, "source_id": "a", "mechanism": "Rubber costs will continue"},
    {"_label": 0, "direction": "precursor",   "polarity": -1, "source_id": "b", "mechanism": "Rubber prices headwind Q4"},
    {"_label": 0, "direction": "consequence",  "polarity": -1, "source_id": "c", "mechanism": "Rubber costs hit Q3 margins"},
])

split = _split_by_direction_polarity(mixed_df, np.eye(3, dtype=np.float32))

print("Before split: all in cluster label 0")
print(mixed_df[["_label", "direction", "source_id"]].to_string(index=False))
print()
print("After split: consequence and precursor are separate sub-clusters")
print(split[["_label", "direction", "source_id"]].to_string(index=False))

## Exercise 4 — Cross-map management ↔ press

After clustering each corpus independently, we find pairs of management and press canonicals that describe the same mechanism. High cosine similarity → `triangulated`. No match → `insider_only` (management only) or `outsider_only` (press only).

In [ ]:
rng = np.random.default_rng(1)

# Logistics: management and press both mention it (triangulated)
logistics_vec = unit(rng.standard_normal(32).astype(np.float32))

# Supply chain risk: only management mentions it (insider_only)
supply_vec = unit(rng.standard_normal(32).astype(np.float32))

# Union dispute: only press mentions it (outsider_only)
union_vec = unit(rng.standard_normal(32).astype(np.float32))

mgmt_centroids = {
    "mgmt-logistics": logistics_vec.copy(),
    "mgmt-supply":    supply_vec.copy(),
}
press_centroids = {
    "press-logistics": unit(logistics_vec + rng.standard_normal(32).astype(np.float32) * 0.01),
    "press-union":     union_vec.copy(),
}

mapping = _cross_map(mgmt_centroids, press_centroids, threshold=0.8)

print("Cross-map results (threshold=0.80):")
for mgmt_id, (press_id, sim) in mapping.items():
    print(f"  {mgmt_id:20s} ↔ {press_id:20s}  sim={sim:.4f}  → triangulated")

unmatched_mgmt = [k for k in mgmt_centroids if k not in mapping]
unmatched_press = [k for k in press_centroids if k not in [v[0] for v in mapping.values()]]

for k in unmatched_mgmt:
    print(f"  {k:20s} (no press match)         → insider_only")
for k in unmatched_press:
    print(f"  {'':20s}   {k:20s}               → outsider_only")

## Exercise 5 — Full cluster_ptcs() on synthetic data

We'll run the complete pipeline on fake embeddings saved to a temp dir — same code path as production.

In [ ]:
rng = np.random.default_rng(7)

def make_cluster(center, n, noise=0.02):
    vecs = [unit(center + rng.standard_normal(center.shape).astype(np.float32) * noise) for _ in range(n)]
    return np.stack(vecs)

# Three management clusters + two press clusters (one shared with management)
c_logistics = unit(rng.standard_normal(32).astype(np.float32))
c_forex      = unit(rng.standard_normal(32).astype(np.float32))
c_rubber     = unit(rng.standard_normal(32).astype(np.float32))
c_union      = unit(rng.standard_normal(32).astype(np.float32))  # press only

mgmt_vecs  = np.vstack([make_cluster(c_logistics, 4), make_cluster(c_forex, 4), make_cluster(c_rubber, 4)])
press_vecs = np.vstack([make_cluster(c_logistics, 4), make_cluster(c_union, 4)])  # logistics shared

all_vecs = np.vstack([mgmt_vecs, press_vecs])
norms = np.linalg.norm(all_vecs, axis=1, keepdims=True)
all_vecs = all_vecs / norms

# Build index DataFrame
rows = []
for i in range(12):   # management
    rows.append({"content_hash": f"hm{i}", "source_id": f"ML.PA-management-2024-01-01-doc{i}",
                 "source_type": "management", "direction": "precursor", "polarity": -1,
                 "source_date": "2024-01-01", "mechanism": f"Management mechanism {i}"})
for i in range(8):    # press
    rows.append({"content_hash": f"hp{i}", "source_id": f"ML.PA-press-2024-01-01-art{i}",
                 "source_type": "press", "direction": "precursor", "polarity": -1,
                 "source_date": "2024-01-01", "mechanism": f"Press mechanism {i}"})

with tempfile.TemporaryDirectory() as tmpdir:
    td = Path(tmpdir)
    np.save(td / "ML.PA_embeddings.npy", all_vecs.astype(np.float32))
    pd.DataFrame(rows).to_csv(td / "ML.PA_embedding_index.csv", index=False)
    out = td / "out"

    canonicals = cluster_ptcs("ML.PA", embeddings_dir=td, output_dir=out,
                               min_cluster_size=3, cross_map_threshold=0.5)

print(canonicals[["canonical_id", "corpus", "direction", "corpus_presence", "n_ptcs"]].to_string())

## The 4-axis typology

Every canonical mechanism carries four independent tags:

In [ ]:
typology = [
    ("Gold",               "triangulated",  "precursor",    "company_specific", "+ significant"),
    ("Privileged",         "insider_only",  "precursor",    "company_specific", "+ significant"),
    ("Management blind spot", "outsider_only", "precursor", "company_specific", "+ significant"),
    ("Sector baseline",    "any",           "any",          "sector_wide",      "any"),
    ("Narrative inflation","insider_only",  "precursor",    "company_specific", "not significant"),
]

header = f"{'Family':<25} {'corpus_presence':<18} {'direction':<12} {'sector_tag':<22} {'statistics'}"
print(header)
print("-" * len(header))
for row in typology:
    print(f"{row[0]:<25} {row[1]:<18} {row[2]:<12} {row[3]:<22} {row[4]}")

---

## Key takeaways

1. Embed with direction+polarity prefix so opposite-stance variants stay apart in vector space.
2. HDBSCAN is noise-tolerant — it won't force every PTC into a cluster.
3. The direction × polarity post-split prevents opposite temporal stances from merging.
4. Cross-mapping at cosine ≥ 0.82 produces the `triangulated` tag — the most trustworthy signal.
5. The 4-axis typology (direction × polarity × corpus_presence × sector_tag) is the core deliverable.

**Next:** Lesson 5 — Time Series: converting the canonical table into testable signals.